# Command Classifier — Raspberry Pi Validation

End-to-end validation of the **31-class command classifier**
(`models/command_classifier/command_classifier.onnx` + `command_metadata.json`),
using the **same production code path** as the app (`commands.CommandClassifier`
and `features.MetadataFrontend`).

### What this notebook measures
| Section | Metrics |
|---|---|
| 1. Environment | OS, CPU, RAM, Raspberry Pi model, CPU temp, ONNX Runtime + providers |
| 2. Model | classes, UNKNOWN threshold, frontend params, input/output shapes, load time |
| 3. Dataset | auto-discovered labeled test set (optional) |
| 4. Latency | frontend / ONNX / end-to-end ms (mean, median, p95, p99), real-time factor, throughput |
| 5. Threads | latency vs `intra_op_num_threads` (1/2/4) |
| 6. Accuracy | top-1/3/5, coverage, per-class precision/recall/F1, confusion matrix, threshold sweep |
| 7. Confidence | top-1 confidence distribution |
| 8. Runtime | sustained run: latency drift + CPU temperature (thermal throttling) |
| 9. Summary | JSON report saved to `results/` |

### Running
* Put this folder **beside the app** (default) or run it from anywhere in the tree —
  the app directory is auto-located via `commands.py` + `models/command_classifier/`.
* Accuracy needs a **labeled test set** (see Section 3). Without one, latency /
  runtime / model metrics still run.
* Only `numpy`, `onnxruntime`, `soundfile` are required. `matplotlib`, `pandas`,
  `psutil` are optional (plots / RAM / nicer tables):

```bash
pip install matplotlib pandas psutil
```

In [ ]:
# ============================================================
# Configuration -- edit these knobs, then run top-to-bottom
# ============================================================
TEST_DATA_DIR     = None      # None = auto-discover; or a dir with <CLASS>/*.wav or a manifest.csv
MANIFEST_CSV      = None      # optional explicit manifest CSV (columns: path,label)
N_WARMUP          = 10        # warm-up inferences before timing
N_RUNS            = 100       # timed inferences per benchmark
THREAD_SWEEP      = [1, 2, 4] # onnxruntime intra-op threads to compare
SUSTAINED_SECONDS = 30        # sustained / thermal test duration
SAMPLE_RATE       = 16000
RESULTS_DIR       = None      # None = ./results beside this notebook

import csv, json, os, platform, statistics, subprocess, sys, time
from collections import Counter
from pathlib import Path

import numpy as np

try:
    from IPython.display import display, Markdown  # noqa: F401
except Exception:  # running outside Jupyter
    def display(*a, **k):
        pass

    def Markdown(*a, **k):
        return ""

try:
    import psutil
    HAVE_PSUTIL = True
except Exception:
    HAVE_PSUTIL = False

try:
    import matplotlib
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

try:
    import pandas as pd
    HAVE_PANDAS = True
except Exception:
    HAVE_PANDAS = False

HERE = Path.cwd()
RESULTS_DIR = Path(RESULTS_DIR) if RESULTS_DIR else (HERE / "results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def find_app_dir(start: Path):
    """Locate the app folder (contains commands.py + the command model)."""
    marker = ("models", "command_classifier", "command_metadata.json")
    for base in [start, *start.parents]:
        if (base / "commands.py").exists() and (base / Path(*marker)).exists():
            return base
    for base in [start, *start.parents]:
        for p in base.glob("*/commands.py"):
            if (p.parent / Path(*marker)).exists():
                return p.parent
    return None


APP_DIR = find_app_dir(HERE)
assert APP_DIR is not None, (
    "Could not locate the app folder. Run this notebook from inside the app tree "
    "(a folder containing commands.py and models/command_classifier/)."
)
sys.path.insert(0, str(APP_DIR))

# ---- Choose the classifier to validate: "MEX2-31class" or "MEX2-trained" ----
CLASSIFIER = "MEX2-trained"
import yaml
_cfg = yaml.safe_load(open(APP_DIR / "config.yaml"))
_reg = ((_cfg.get("command", {}) or {}).get("models", {}) or {}) or {"default": "models/command_classifier"}
_k = next((k for k in _reg if CLASSIFIER.lower() in (k.lower(), Path(_reg[k]).name.lower())), None)
assert _k, f"Unknown CLASSIFIER {CLASSIFIER!r}; options: {list(_reg)}"
MODEL_DIR = APP_DIR / _reg[_k]
ONNX_PATH = next((MODEL_DIR / f for f in ("best_model.onnx", "command_classifier.onnx")
                  if (MODEL_DIR / f).exists()), MODEL_DIR / "best_model.onnx")
print("classifier:", _k, "->", MODEL_DIR)
META_PATH = MODEL_DIR / "command_metadata.json"

print("app dir   :", APP_DIR)
print("model dir :", MODEL_DIR)
print("results   :", RESULTS_DIR)
print("optional  : matplotlib=%s pandas=%s psutil=%s" % (HAVE_MPL, HAVE_PANDAS, HAVE_PSUTIL))

## 1. Runtime environment

Captures the platform fingerprint that matters for an edge deployment
(Raspberry Pi model, CPU temperature, RAM, ONNX Runtime providers).

In [ ]:
def _read_first(paths):
    for p in paths:
        try:
            return Path(p).read_bytes().decode(errors="ignore").strip("\x00").strip()
        except Exception:
            pass
    return None


def rpi_model():
    return _read_first(["/proc/device-tree/model", "/sys/firmware/devicetree/base/model"])


def cpu_temp_c():
    v = _read_first(["/sys/class/thermal/thermal_zone0/temp"])
    if v and v.strip().isdigit():
        return int(v.strip()) / 1000.0
    try:
        out = subprocess.check_output(["vcgencmd", "measure_temp"],
                                      stderr=subprocess.DEVNULL).decode()
        return float(out.strip().split("=")[1].split("'")[0])
    except Exception:
        return None


def mem_total_mb():
    if HAVE_PSUTIL:
        return psutil.virtual_memory().total / 1e6
    v = _read_first(["/proc/meminfo"])
    if v:
        for line in v.splitlines():
            if line.startswith("MemTotal:"):
                return int(line.split()[1]) / 1024.0
    return None


import onnxruntime as ort

ENV = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "cpu_count": os.cpu_count(),
    "rpi_model": rpi_model(),
    "cpu_temp_c": cpu_temp_c(),
    "ram_mb": round(mem_total_mb(), 1) if mem_total_mb() else None,
    "onnxruntime": ort.__version__,
    "providers": ort.get_available_providers(),
    "numpy": np.__version__,
    "onnx_file_mb": round(ONNX_PATH.stat().st_size / 1e6, 2),
}
for k, v in ENV.items():
    print(f"{k:14s}: {v}")

## 2. Model & metadata

Reads the frontend parameters, class list and calibrated UNKNOWN threshold from
`command_metadata.json`, then instantiates the **production** classifier.

In [ ]:
# ---- 2. Model + frontend (works with BOTH classifier formats) ----
CMD_META = MODEL_DIR / "command_metadata.json"
CMD_LABELS = MODEL_DIR / "labels.json"

if CMD_META.exists():                         # bundled / metadata format
    META = json.load(open(CMD_META))
    CLASSES = [None] * len(META["class_to_idx"])
    for name, i in META["class_to_idx"].items():
        CLASSES[int(i)] = name
    _thr = float(META.get("unknown_threshold", 0.5))
    FE = {k: META[k] for k in ("sr", "n_fft", "hop", "n_mels", "max_frames", "log_eps") if k in META}
else:                                         # trained / frontend.json format
    _lab = json.load(open(CMD_LABELS))
    CLASSES = [_lab["id_to_class"][str(i)] for i in range(len(_lab["id_to_class"]))]
    _fe = json.load(open(MODEL_DIR / "frontend.json"))
    _thr = 0.5
    FE = {"sr": _fe["sample_rate"], "n_fft": _fe["n_fft"], "hop": _fe["hop_length"],
          "n_mels": _fe["n_mels"], "max_frames": None, "log_eps": None}

from commands import CommandClassifier
t0 = time.perf_counter()
clf = CommandClassifier(MODEL_DIR, intents=CLASSES, unknown_threshold=_thr)
LOAD_MS = (time.perf_counter() - t0) * 1e3
assert clf.real, "classifier ONNX not found / not loaded"

CLASSES = list(clf.intents)
NUM_CLASSES = len(CLASSES)
UNKNOWN_THRESHOLD = float(clf.unknown_threshold)

_raw_fe = clf.frontend
SR = int(getattr(_raw_fe, "sr", getattr(_raw_fe, "sample_rate", SAMPLE_RATE)))
WINDOW_SAMPLES = int(getattr(_raw_fe, "window_samples", None) or getattr(_raw_fe, "n_samples", None)
                     or int(SR * getattr(_raw_fe, "clip_duration", 2.0)))
CLIP_S = WINDOW_SAMPLES / SR


def fe(w):                      # always (1, 1, n_mels, T) for a direct sess.run()
    y = np.asarray(_raw_fe(w), dtype=np.float32)
    return y[None, None] if y.ndim == 2 else y

print("classes           :", NUM_CLASSES)
print("unknown_threshold :", UNKNOWN_THRESHOLD)
print("frontend          :", clf.frontend_kind, FE)
print(f"window_samples    : {WINDOW_SAMPLES}  ({CLIP_S*1000:.1f} ms @ {SR} Hz)")
print("classes           :", CLASSES)

In [ ]:
from commands import CommandClassifier   # production wrapper from the app

t0 = time.perf_counter()
clf = CommandClassifier(MODEL_DIR, intents=CLASSES, unknown_threshold=UNKNOWN_THRESHOLD)
LOAD_MS = (time.perf_counter() - t0) * 1e3
assert clf.real, "command_classifier.onnx was not found / not loaded"

_in = clf.sess.get_inputs()[0]
_out = clf.sess.get_outputs()[0]
MODEL = {
    "onnx_file_mb": round(ONNX_PATH.stat().st_size / 1e6, 2),
    "load_ms": LOAD_MS,
    "input": {"name": _in.name, "shape": _in.shape, "type": _in.type},
    "output": {"name": _out.name, "shape": _out.shape, "type": _out.type},
    "classes": NUM_CLASSES,
    "unknown_threshold": clf.unknown_threshold,
}
print(f"model load        : {LOAD_MS:.1f} ms")
print(f"input             : {_in.name} {_in.shape} {_in.type}")
print(f"output            : {_out.name} {_out.shape} {_out.type}")
print(f"classes / thr     : {len(clf.intents)} / {clf.unknown_threshold}")

## 3. Labeled test set (optional)

Accuracy metrics need ground-truth labels. This cell auto-discovers a test set
from several layouts (first match wins):

* an explicit `TEST_DATA_DIR` / `MANIFEST_CSV` (edit above),
* `./test_data/<CLASS>/*.wav` (folder per class),
* `./test_data/manifest.csv` with columns `path,label`,
* `<app>/data/command/test/<CLASS>/*.wav`, `<app>/test_data/...`, etc.

Labels must match the model's `class_to_idx`. Without a dataset the accuracy
sections are skipped and only latency / runtime metrics are produced.

In [ ]:
from features import read_wav_mono_16k


def load_wav(path, sr=SAMPLE_RATE):
    try:
        return read_wav_mono_16k(path, sr)
    except Exception:
        import soundfile as sf
        data, s = sf.read(str(path), dtype="float32", always_2d=True)
        return data.mean(axis=1).astype(np.float32)

import re

NEG_DIR_NAMES = {"_negative", "negatives", "negative", "unknown", "background",
                 "noise", "noncommand", "non_command", "out_of_scope", "outofscope", "oos"}

# (command, slot_value) -> model class, for the slot intents
SLOT_MAP = {
    ("ALARM", "6:00 AM"): "ALARM_6_00AM",
    ("ALARM", "8:00 AM"): "ALARM_8_00AM",
    ("ALARM", "9:00 PM"): "ALARM_9_00PM",
    ("BRIGHTNESS", "20 percent"): "BRIGHTNESS_20",
    ("BRIGHTNESS", "60 percent"): "BRIGHTNESS_60",
    ("BRIGHTNESS", "100 percent"): "BRIGHTNESS_100",
    ("COLOR", "Red"): "COLOR_RED",
    ("COLOR", "Green"): "COLOR_GREEN",
    ("COLOR", "Blue"): "COLOR_BLUE",
    ("COLOR", "Yellow"): "COLOR_YELLOW",
    ("CREATE_REMINDER", "Drink water"): "CREATE_REMINDER_DRINK_WATER",
    ("CREATE_REMINDER", "Exercise"): "CREATE_REMINDER_EXERCISE",
    ("CREATE_REMINDER", "Study"): "CREATE_REMINDER_STUDY",
    ("TEMPERATURE", "18 degrees"): "TEMPERATURE_18",
    ("TEMPERATURE", "22 degrees"): "TEMPERATURE_22",
    ("TEMPERATURE", "26 degrees"): "TEMPERATURE_26",
    ("TIMER", "10 seconds"): "TIMER_10s",
    ("TIMER", "30 seconds"): "TIMER_30s",
    ("TIMER", "1 minute"): "TIMER_1m",
}


def _norm(x):
    return re.sub(r"[^a-z0-9]", "", str(x or "").lower())


def resolve_label(command, slot_value=""):
    """Map a manifest (command[, slot_value]) to one of the 31 model classes."""
    cmd = str(command or "").strip()
    slot = str(slot_value or "").strip()
    if cmd in CLASSES:
        return cmd
    cands = []
    if (cmd, slot) in SLOT_MAP:
        cands.append(SLOT_MAP[(cmd, slot)])
    for (c, s), cls in SLOT_MAP.items():                 # tolerant of case/format
        if c == cmd and _norm(s) == _norm(slot):
            cands.append(cls)
    cands += [f"{cmd}_{slot}".replace(" ", "_").upper(), f"{cmd}{slot}".upper()]
    nums = re.findall(r"\d+", slot)
    if nums:
        cands.append(f"{cmd}_{nums[0]}")
    cands.append(cmd.upper())
    for c in cands:
        if c in CLASSES:
            return c
    for c in cands:                                       # normalized match
        for cls in CLASSES:
            if _norm(cls) == _norm(c):
                return cls
    return None


def _pick(row, *names):
    for n in names:
        for k in row:
            if k.lower() == n:
                return k
    return None


def read_manifest(m):
    """Manifest (any schema) -> (positives[(path,class)], negatives[path], unresolved)."""
    with open(m, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    pos, neg, bad = [], [], []
    if not rows:
        return pos, neg, bad
    fk = _pick(rows[0], "file", "path", "wav", "filename")
    ck = _pick(rows[0], "command", "label", "class", "intent")
    sk = _pick(rows[0], "slot_value", "slot", "value")
    ok = _pick(rows[0], "out_of_scope", "outofscope", "oos", "is_negative", "negative")
    for r in rows:
        p = (r.get(fk) or "").strip() if fk else ""
        if not p:
            continue
        pp = Path(p)
        if not pp.is_absolute():
            pp = m.parent / pp
        if not pp.exists():                              # e.g. files under an audio/ subdir
            alt = m.parent / "audio" / Path(p).name
            if alt.exists():
                pp = alt
        cmd = (r.get(ck) or "").strip() if ck else ""
        slot = (r.get(sk) or "").strip() if sk else ""
        is_neg = (str(r.get(ok, "")).strip().lower() in ("1", "true", "yes") if ok else False) \
            or _norm(cmd) in {_norm(x) for x in NEG_DIR_NAMES}
        if is_neg:
            neg.append(pp)
            continue
        cls = resolve_label(cmd, slot)
        if cls:
            pos.append((pp, cls))
        else:
            bad.append((cmd, slot))
    return pos, neg, bad


def _roots(explicit):
    r = []
    if explicit:
        r.append(Path(explicit))
    r += [HERE / "test_data", APP_DIR / "data" / "command" / "test",
          APP_DIR / "data" / "command", APP_DIR / "test_data",
          APP_DIR / "data", APP_DIR.parent / "data" / "command" / "test"]
    return [x for x in r if x and x.exists()]


def _find_manifests(roots, explicit=None):
    ms = [Path(explicit)] if explicit else []
    for r in roots:
        ms += sorted(r.glob("*.csv")) + sorted(r.glob("*/manifest*.csv")) + sorted(r.glob("*/*.csv"))
    seen, out = set(), []
    for m in ms:
        if m.exists() and str(m.resolve()) not in seen:
            seen.add(str(m.resolve()))
            out.append(m)
    return out


# ---- discover labeled dataset (holdout manifest first) ----
POS, NEG_PATHS, BAD = [], [], []
DATASET_SRC = "none found"

for m in _find_manifests(_roots(TEST_DATA_DIR), MANIFEST_CSV):
    p, n, b = read_manifest(m)
    if p or n:
        POS, NEG_PATHS, BAD = p, n, b
        DATASET_SRC = f"manifest {m}"
        break

if not POS:                                     # fallback: folder-per-class
    for r in _roots(TEST_DATA_DIR):
        items = [(w, sub.name)
                 for sub in sorted(d for d in r.iterdir() if d.is_dir())
                 for w in sorted(sub.glob("*.wav"))]
        if items:
            POS, DATASET_SRC = items, f"folder-per-class {r}"
            break
if not POS:                                     # fallback: flat <LABEL>__file.wav
    for r in _roots(TEST_DATA_DIR):
        items = [(w, w.stem.split("__")[0]) for w in sorted(r.glob("*.wav"))]
        if items:
            POS, DATASET_SRC = items, f"flat prefix {r}"
            break

DATASET = POS
VALID = [(p, l) for p, l in DATASET if l in CLASSES]
BAD_LABELS = sorted({l for _, l in DATASET if l not in CLASSES})
print("dataset source :", DATASET_SRC)
print("samples        :", len(DATASET), f"(valid={len(VALID)}, other labels={BAD_LABELS})")
print("negatives      :", len(NEG_PATHS))
if DATASET:
    print("per-class      :", dict(sorted(Counter(l for _, l in DATASET).items())))
else:
    print("No labeled data found -> accuracy sections will be skipped.")
if BAD:
    print("unresolved     :", BAD[:8], f"(+{len(BAD)-8} more)" if len(BAD) > 8 else "")


## 4. Latency & throughput

Timed separately so you can see where time goes:

* **frontend** — NumPy log-mel feature extraction,
* **ONNX** — pure `session.run` on pre-computed features,
* **end-to-end** — exactly what the app does per command (`predict`).

*Real-time factor* = end-to-end time ÷ clip length; `< 1` means faster than real time.

In [ ]:
def stats(times):
    t = np.asarray(times, dtype=float)
    return {"mean_ms": float(t.mean()), "median_ms": float(np.median(t)),
            "p95_ms": float(np.percentile(t, 95)), "p99_ms": float(np.percentile(t, 99)),
            "min_ms": float(t.min()), "max_ms": float(t.max()), "std_ms": float(t.std())}


def bench_frontend(clf, wavs, warm, runs):
    fe = clf.frontend
    for i in range(warm):
        fe(wavs[i % len(wavs)])
    ts = []
    for i in range(runs):
        w = wavs[i % len(wavs)]
        t = time.perf_counter_ns(); fe(w); ts.append((time.perf_counter_ns() - t) / 1e6)
    return ts


def bench_onnx(clf, mels, warm, runs):
    # Handle both frontend formats:
    # - MetadataFrontend (bundled): mels are already 4D [1, 1, n_mels, T]
    # - LogMelFrontend (trained): mels are 2D [n_mels, T], need to add batch+channel dims
    def _prep(m):
        if m.ndim == 2:
            return m[None, None]  # [n_mels, T] -> [1, 1, n_mels, T]
        return m
    for i in range(warm):
        clf.sess.run(None, {clf.input_name: _prep(mels[i % len(mels)])})
    ts = []
    for i in range(runs):
        m = mels[i % len(mels)]
        t = time.perf_counter_ns()
        clf.sess.run(None, {clf.input_name: _prep(m)})
        ts.append((time.perf_counter_ns() - t) / 1e6)
    return ts


def bench_e2e(clf, wavs, warm, runs):
    for i in range(warm):
        clf.predict(wavs[i % len(wavs)])
    ts = []
    for i in range(runs):
        w = wavs[i % len(wavs)]
        t = time.perf_counter_ns(); clf.predict(w); ts.append((time.perf_counter_ns() - t) / 1e6)
    return ts

In [ ]:
# Pick benchmarking audio: dataset > bundled wake samples > silence
if DATASET:
    BENCH_WAVS = [load_wav(p) for p, _ in DATASET[:min(len(DATASET), 20)]]
    BENCH_SRC = "dataset"
else:
    samp = sorted((APP_DIR / "samples").glob("*.wav"))
    BENCH_WAVS = [load_wav(p) for p in samp[:5]] or [np.zeros(WINDOW_SAMPLES, np.float32)]
    BENCH_SRC = "bundled samples" if samp else "silence"

MELS = [clf.frontend(w) for w in BENCH_WAVS]

fe_t = bench_frontend(clf, BENCH_WAVS, N_WARMUP, N_RUNS)
onnx_t = bench_onnx(clf, MELS, N_WARMUP, N_RUNS)
e2e_t = bench_e2e(clf, BENCH_WAVS, N_WARMUP, N_RUNS)

LATENCY = {"bench_source": BENCH_SRC, "clip_ms": CLIP_S * 1000,
           "frontend": stats(fe_t), "onnx": stats(onnx_t), "end_to_end": stats(e2e_t)}
e2e = LATENCY["end_to_end"]
RTF = e2e["mean_ms"] / (CLIP_S * 1000)
LATENCY["real_time_factor"] = RTF
LATENCY["throughput_per_s"] = 1000.0 / e2e["mean_ms"]

print(f"bench source       : {BENCH_SRC} ({len(BENCH_WAVS)} clips)")
print(f"clip length        : {CLIP_S*1000:.1f} ms")
print(f"frontend   mean/p95: {LATENCY['frontend']['mean_ms']:.2f} / {LATENCY['frontend']['p95_ms']:.2f} ms")
print(f"onnx       mean/p95: {LATENCY['onnx']['mean_ms']:.2f} / {LATENCY['onnx']['p95_ms']:.2f} ms")
print(f"end-to-end mean/p95: {e2e['mean_ms']:.2f} / {e2e['p95_ms']:.2f} ms")
print(f"end-to-end median  : {e2e['median_ms']:.2f} ms   p99: {e2e['p99_ms']:.2f} ms")
print(f"real-time factor   : {RTF:.4f}  (x{1/RTF:.1f} faster than real time)")
print(f"throughput         : {LATENCY['throughput_per_s']:.1f} inferences/s")

## 5. Thread sweep

The app caps ONNX Runtime to 2 intra-op threads for Pi thermals. This compares
latency at 1 / 2 / 4 threads so you can pick the sweet spot on the target board.

In [ ]:
max_threads = os.cpu_count() or 1
sweep = {}
for nt in sorted(set(t for t in THREAD_SWEEP if t <= max_threads) | {min(2, max_threads)}):
    c = CommandClassifier(MODEL_DIR, intents=CLASSES, num_threads=nt,
                          unknown_threshold=UNKNOWN_THRESHOLD)
    s = stats(bench_e2e(c, BENCH_WAVS, max(3, N_WARMUP // 2), max(20, N_RUNS // 2)))
    s["throughput_per_s"] = 1000.0 / s["mean_ms"]
    sweep[int(nt)] = s
    print(f"threads={nt}: mean={s['mean_ms']:6.2f} ms  p95={s['p95_ms']:6.2f} ms  "
          f"({s['throughput_per_s']:5.1f}/s)")
LATENCY["thread_sweep"] = sweep

## 6. Accuracy

Computed only when a labeled test set was found. Reports top-1/3/5 accuracy,
coverage vs the UNKNOWN threshold, and per-class precision/recall/F1.

* **coverage** = fraction of clips whose top-1 confidence ≥ `unknown_threshold`
  (i.e. accepted, not rejected as UNKNOWN),
* **accuracy on covered** = accuracy among the accepted clips.

In [ ]:
ACC = None
PROBS = None
ytrue = ypred = conf = rejected = None

if VALID:
    X, y = [], []
    for p, lab in VALID:
        try:
            X.append(load_wav(p))
            y.append(CLASSES.index(lab))
        except Exception as e:
            print("skip", p, e)
    ytrue = np.asarray(y)
    PROBS = np.vstack([clf.predict(w) for w in X])
    ypred = PROBS.argmax(1)
    conf = PROBS.max(1)
    rejected = conf < UNKNOWN_THRESHOLD

    top1 = float((ypred == ytrue).mean())

    def topk(k):
        order = np.argsort(-PROBS, axis=1)[:, :k]
        return float(np.mean([ytrue[i] in order[i] for i in range(len(ytrue))]))

    coverage = float((~rejected).mean())
    acc_cov = float((ypred[~rejected] == ytrue[~rejected]).mean()) if (~rejected).any() else float("nan")

    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    for t, p in zip(ytrue, ypred):
        cm[t, p] += 1
    per_class = {}
    for i, name in enumerate(CLASSES):
        tp = int(cm[i, i]); fp = int(cm[:, i].sum()) - tp; fn = int(cm[i, :].sum()) - tp
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        per_class[name] = {"support": int(cm[i, :].sum()), "precision": prec,
                           "recall": rec, "f1": f1}

    ACC = {"n": int(len(ytrue)), "top1": top1, "top3": topk(3), "top5": topk(5),
           "coverage": coverage, "accuracy_on_covered": acc_cov,
           "unknown_rate": float(rejected.mean()),
           "macro_f1": float(np.mean([v["f1"] for v in per_class.values()])),
           "per_class": per_class, "confusion": cm.tolist()}
    print(f"samples            : {ACC['n']}")
    print(f"top-1 accuracy     : {top1*100:.2f}%")
    print(f"top-3 / top-5      : {ACC['top3']*100:.2f}% / {ACC['top5']*100:.2f}%")
    print(f"coverage (>=thr)   : {coverage*100:.2f}%   (UNKNOWN rate {rejected.mean()*100:.2f}%)")
    print(f"acc on covered     : {acc_cov*100:.2f}%")
    print(f"macro F1           : {ACC['macro_f1']*100:.2f}%")
else:
    print("Accuracy skipped (no labeled dataset).")

# --- class index <-> intent name (so numeric confusion-matrix axes are readable) ---
if ACC is not None:
    print("\nclass index -> intent:")
    print("  " + "  ".join(f"{i}:{n}" for i, n in enumerate(CLASSES)))
    _cm = np.asarray(ACC["confusion"])
    _off = [(int(_cm[t, p]), t, p) for t in range(len(CLASSES))
            for p in range(len(CLASSES)) if t != p and _cm[t, p]]
    if _off:
        _off.sort(reverse=True)
        print("\ntop misclassifications (true -> predicted : count):")
        for _c, _t, _p in _off[:8]:
            print(f"  {CLASSES[_t]:30s} -> {CLASSES[_p]:30s} : {_c}")
    _col = _cm.sum(0)
    print("\nmost-predicted classes (large column = over-prediction):")
    for _i in np.argsort(-_col)[:6]:
        print(f"  [{_i}] {CLASSES[_i]:30s} {int(_col[_i])}")


In [ ]:
if ACC:
    print(f"{'class':30s} {'n':>4s} {'prec':>8s} {'rec':>8s} {'f1':>8s}")
    for name, m in ACC["per_class"].items():
        print(f"{name:30s} {m['support']:4d} {m['precision']*100:7.1f}% "
              f"{m['recall']*100:7.1f}% {m['f1']*100:7.1f}%")

    print("\nUNKNOWN-threshold sweep (coverage vs accuracy on accepted clips):")
    rows = []
    for thr in [0.0, 0.05, 0.1, round(UNKNOWN_THRESHOLD, 3), 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        rej = conf < thr
        cov = float((~rej).mean())
        acc_c = float((ypred[~rej] == ytrue[~rej]).mean()) if (~rej).any() else float("nan")
        rows.append([float(thr), cov, acc_c])
    print(f"{'thr':>6s} {'coverage':>10s} {'acc@cov':>10s}")
    for thr, cov, acc_c in rows:
        print(f"{thr:6.3f} {cov*100:9.2f}% {acc_c*100:9.2f}%")
    ACC["threshold_sweep"] = rows
else:
    print("skipped (no labeled dataset)")

## 7. Confidence & plots

Distribution of the top-1 softmax probability (calibration sanity) plus latency
histograms. Plots are drawn only if `matplotlib` is installed.

In [ ]:
PLOT_FILES = []
if ACC:
    print(f"top-1 confidence : mean={conf.mean():.3f} median={np.median(conf):.3f} "
          f"min={conf.min():.3f} max={conf.max():.3f}")
    if HAVE_MPL:
        fig, axs = plt.subplots(1, 2, figsize=(16, 5))
        axs[0].hist(conf, bins=20, color="#3ec46d"); axs[0].set_title("top-1 confidence")
        axs[0].set_xlabel("probability")
        axs[1].hist(e2e_t, bins=20, color="#4a90d9"); axs[1].set_title("end-to-end latency (ms)")
        plt.tight_layout()
        p = RESULTS_DIR / "accuracy_plots.png"; plt.savefig(p, dpi=120); plt.show()
        PLOT_FILES.append(str(p))

        cm = np.asarray(ACC["confusion"])
        fig, ax = plt.subplots(figsize=(9, 8))
        im = ax.imshow(cm, cmap="Blues")
        k = len(CLASSES)
        ax.set_xticks(range(k))
        ax.set_xticklabels(CLASSES, rotation=90, fontsize=6)
        ax.set_yticks(range(k))
        ax.set_yticklabels(CLASSES, fontsize=6)
        ax.set_xlabel("predicted")
        ax.set_ylabel("true")
        ax.set_title("confusion matrix")
        fig.colorbar(im, ax=ax, fraction=0.046)
        plt.tight_layout()
        p = RESULTS_DIR / "accuracy_confusion_matrix.png"; plt.savefig(p, dpi=120); plt.show()
        PLOT_FILES.append(str(p))
    else:
        print("(install matplotlib for plots)")
else:
    print("no labeled dataset -> confidence stats skipped")
    if HAVE_MPL:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].hist(e2e_t, bins=20, color="#4a90d9"); ax[0].set_title("end-to-end latency (ms)")
        ax[1].hist(fe_t, bins=20, color="#e0a52b"); ax[1].set_title("frontend latency (ms)")
        plt.tight_layout()
        p = RESULTS_DIR / "latency_plots.png"; plt.savefig(p, dpi=120); plt.show()
        PLOT_FILES.append(str(p))

## 8. Sustained runtime & thermal stability

Runs back-to-back inference for `SUSTAINED_SECONDS`, sampling CPU temperature
every second. A growing latency between the first and second half signals
thermal throttling — the key risk for an always-on Pi 5 workload.

In [ ]:
def sustained_test(clf, wavs, seconds, temp_fn, temp_every=1.0):
    t_end = time.time() + seconds
    lat, temps, stamps = [], [], []
    next_temp = time.time()
    t_start = temp_fn() if temp_fn else None
    while time.time() < t_end:
        w = wavs[len(lat) % len(wavs)]
        t = time.perf_counter_ns()
        clf.predict(w)
        lat.append((time.perf_counter_ns() - t) / 1e6)
        now = time.time()
        if temp_fn and now >= next_temp:
            tv = temp_fn()
            if tv is not None:
                temps.append(tv); stamps.append(now - t_end + seconds)
            next_temp = now + temp_every
    return np.asarray(lat), temps, stamps, t_start, (temp_fn() if temp_fn else None)


lat_s, temps, stamps, temp0, temp1 = sustained_test(clf, BENCH_WAVS, SUSTAINED_SECONDS, cpu_temp_c)
half = max(1, len(lat_s) // 2)
first, second = float(lat_s[:half].mean()), float(lat_s[half:].mean())
drift_pct = (second / first - 1) * 100 if first else 0.0

RUNTIME = {"sustained_seconds": SUSTAINED_SECONDS, "n_inferences": int(len(lat_s)),
           "mean_ms": float(lat_s.mean()), "p95_ms": float(np.percentile(lat_s, 95)),
           "first_half_ms": first, "second_half_ms": second, "drift_ms": second - first,
           "drift_pct": drift_pct,
           "temp_start_c": temp0, "temp_end_c": temp1,
           "temp_max_c": (max(temps) if temps else None), "temp_series": temps}

print(f"sustained run      : {len(lat_s)} inferences in {SUSTAINED_SECONDS}s")
print(f"latency mean/p95   : {lat_s.mean():.2f} / {np.percentile(lat_s, 95):.2f} ms")
print(f"first half         : {first:.2f} ms")
print(f"second half        : {second:.2f} ms  (drift {second-first:+.2f} ms / {drift_pct:+.1f}%)")
if temp0 is not None:
    print(f"cpu temp start/end : {temp0:.1f} / {temp1:.1f} C   max {max(temps):.1f} C")
else:
    print("cpu temp           : n/a (not a Raspberry Pi / no sensor)")

if HAVE_MPL and stamps:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(lat_s, color="#4a90d9"); ax[0].set_title("latency over time (ms)")
    ax[0].set_xlabel("inference #")
    ax[1].plot(stamps, temps, "-o", color="#e0483a"); ax[1].set_title("cpu temp (C)")
    ax[1].set_xlabel("seconds")
    plt.tight_layout()
    p = RESULTS_DIR / "sustained_plots.png"; plt.savefig(p, dpi=120); plt.show()
    PLOT_FILES.append(str(p))

## 9. Summary & report

Aggregates every metric and writes a machine-readable report to
`results/command_classifier_validation_mex2_trained.json` (plus optional CSVs / PNG plots).

In [ ]:
SUMMARY = {
    "environment": ENV,
    "model": MODEL,
    "dataset": {"source": DATASET_SRC, "n": len(DATASET), "valid": len(VALID)},
    "latency": LATENCY,
    "accuracy": ACC,
    "runtime": RUNTIME,
    "plots": PLOT_FILES,
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

report = RESULTS_DIR / "command_classifier_validation_mex2_trained.json"
with open(report, "w") as f:
    json.dump(SUMMARY, f, indent=2, default=float)

if ACC and HAVE_PANDAS:
    pd.DataFrame(ACC["per_class"]).T.to_csv(RESULTS_DIR / "per_class_metrics.csv")
if HAVE_PANDAS:
    pd.DataFrame([{"threads": k, **v} for k, v in LATENCY["thread_sweep"].items()]).to_csv(
        RESULTS_DIR / "thread_sweep.csv", index=False)

print("============================ SUMMARY ============================")
print(f"target             : {ENV.get('rpi_model') or ENV['platform']}")
print(f"model             : {ENV['onnx_file_mb']} MB, {NUM_CLASSES} classes, "
      f"thr={UNKNOWN_THRESHOLD}")
print(f"end-to-end latency: {LATENCY['end_to_end']['mean_ms']:.2f} ms "
      f"(p95 {LATENCY['end_to_end']['p95_ms']:.2f} ms)")
print(f"real-time factor  : {LATENCY['real_time_factor']:.4f}")
print(f"throughput        : {LATENCY['throughput_per_s']:.1f} inferences/s")
if ACC:
    print(f"top-1 / top-5 acc : {ACC['top1']*100:.2f}% / {ACC['top5']*100:.2f}%")
    print(f"coverage          : {ACC['coverage']*100:.2f}%  "
          f"(acc on covered {ACC['accuracy_on_covered']*100:.2f}%)")
    print(f"macro F1          : {ACC['macro_f1']*100:.2f}%")
print(f"sustained drift   : {RUNTIME['drift_ms']:+.2f} ms ({RUNTIME['drift_pct']:+.1f}%)")
if RUNTIME["temp_max_c"] is not None:
    print(f"max cpu temp      : {RUNTIME['temp_max_c']:.1f} C")
print(f"report            : {report}")
print("================================================================")

---

### Notes & tips
* **Run top-to-bottom**; the load and warm-up cells matter for realistic timing.
* For a clean latency number, stop other CPU-heavy processes on the Pi first.
* Launch Jupyter from this folder so relative paths resolve:
  `jupyter notebook` then open `command_classifier_validation.ipynb`, or
  `jupyter nbconvert --execute --to notebook --inplace command_classifier_validation.ipynb`.
* Re-run the notebook once the ordered test set is in place to fill the accuracy
  section; the JSON report makes it easy to diff runs across boards / cooling setups.